In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
claims_df = spark.table("catalog1.silver_layer.silver_claim_table")


In [0]:
import re

def extract_claim_with_procedures(text):

    claim_id = re.search(r"Claim ID:\s*(\w+)", text)
    patient_id = re.search(r"Patient ID:\s*(\w+)", text)
    policy = re.search(r"Policy:\s*(.*?)(?=\s{2,}|$)", text)
    diagnosis = re.search(r"Diagnosis:\s*([A-Za-z\s]+)", text)
    admission = re.search(r"Admission Date:\s*([0-9A-Za-z\-]+)", text)
    discharge = re.search(r"Discharge Date:\s*([0-9A-Za-z\-]+)", text)
    total = re.search(r"Total Claim:\s*\$(\d+)", text)

    # Extract procedures (multiple)
    procedures = re.findall(r"Procedure:\s*([^:]+):\s*\$(\d+)", text)

    rows = []

    for proc in procedures:
        rows.append((
            claim_id.group(1) if claim_id else None,
            patient_id.group(1) if patient_id else None,
            policy.group(1).strip() if policy else None,
            diagnosis.group(1).strip() if diagnosis else None,
            proc[0].strip(),              # procedure name
            int(proc[1]),                 # procedure cost
            int(total.group(1)) if total else None,
            admission.group(1) if admission else None,
            discharge.group(1) if discharge else None
        ))

    return rows

rows = claims_df.collect()

structured_claims = []

for row in rows:
    structured_claims.extend(extract_claim_with_procedures(row.clean_text))

In [0]:
gold_claim_df = spark.createDataFrame(
    structured_claims,
    ["claim_id",
     "patient_id",
     "policy_name",
     "diagnosis",
     "procedure",
     "procedure_cost",
     "total_amount",
     "admission_date",
     "discharge_date"]
)

In [0]:
gold_claim_df.display()

In [0]:
%sql
create database if not exists catalog1.gold_layer;

In [0]:
gold_claim_df.write.\
    format("delta") \
    .mode("overwrite") \
    .saveAsTable("catalog1.gold_layer.gold_claims_table")

In [0]:
policy_df=spark.table("catalog1.silver_layer.silver_policy_table")
policy_df.display()

In [0]:
import re

def extract_policy_fields(text):
    name = re.search(r"Policy Name:\s*(.+?)\s*Policy ID:", text)
    policy_id = re.search(r"Policy ID:\s*([A-Z0-9\-]+)", text)

    yearly_limit = re.search(r"Hospitalization up to \$([\d,]+)", text)

    icu_limit = re.search(r"ICU:\s*\$([\d,]+)", text)

    daily_limit = re.search(r"Hospital Stay:\s*\$([\d,]+)\s*per day", text)
    max_days = re.search(r"max\s*(\d+)\s*days", text)

    waiting = re.search(r"Waiting Period:\s*(.+?)\n", text)

    icu_covered = "ICU Coverage" in text

    return (
        name.group(1).strip() if name else None,
        policy_id.group(1) if policy_id else None,
        int(yearly_limit.group(1).replace(",", "")) if yearly_limit else None,
        int(icu_limit.group(1).replace(",", "")) if icu_limit else None,
        int(daily_limit.group(1).replace(",", "")) if daily_limit else None,
        int(max_days.group(1)) if max_days else None,
        waiting.group(1).strip() if waiting else "No waiting period",
        icu_covered
    )

In [0]:
rows = policy_df.collect()

structured_policies = []

for row in rows:
    structured_policies.append(extract_policy_fields(row.clean_text))

In [0]:
gold_policy_df = spark.createDataFrame(
    structured_policies,
    ["policy_name",
     "policy_id",
     "yearly_limit",
     "icu_daily_limit",
     "daily_room_limit",
     "max_days",
     "waiting_period",
     "icu_covered"]
)

In [0]:
gold_policy_df.display()

In [0]:
gold_policy_df.write.\
    format("delta") \
    .mode("overwrite") \
    .saveAsTable("catalog1.gold_layer.gold_policy_table")

In [0]:
claims = spark.table("catalog1.gold_layer.gold_claims_table")
pricing = spark.table("catalog1.bronze_layer.bronze_cost_table")

joined_df = claims.join(
    pricing,
    claims.procedure == pricing.Procedure,
    "left"
)

In [0]:
joined_df.display()

In [0]:
from pyspark.sql.functions import col, when

fraud_df = joined_df.withColumn(
    "overpricing_fraud",
    when(
        col("procedure_cost") > col("Average_Cost") * 1.30,
        True
    ).otherwise(False)
)

In [0]:
fraud_df = fraud_df.withColumn(
    "risk_level",
    when(col("overpricing_fraud") == True, "LOW")
    .otherwise("NONE")
)

In [0]:
from pyspark.sql.functions import round

fraud_df = fraud_df.withColumn(
    "price_difference",
    col("procedure_cost") - col("Average_Cost")
)

fraud_df = fraud_df.withColumn(
    "allowed_limit",
    col("Average_Cost") * 1.30
)

fraud_df = fraud_df.withColumn(
    "overpricing_percentage",
    round(
        (col("procedure_cost") - col("Average_Cost")) / col("Average_Cost") * 100,
        2
    )
)

In [0]:
fraud_df = fraud_df.withColumn(
    "fraud_reason",
    when(
        col("overpricing_fraud") == True,
        "Procedure cost exceeds 130% of average cost."
    ).otherwise(
        "Procedure cost is within acceptable range."
    )
)

In [0]:
fraud_df.display()

In [0]:
gold_fraud_df = fraud_df.select(
    "claim_id",
    "patient_id",
    "procedure_cost",
    "Average_Cost",
    "allowed_limit",
    "price_difference",
    "overpricing_percentage",
    "overpricing_fraud",
    "fraud_reason"
)


In [0]:
gold_fraud_df.write.\
    format("delta") \
    .mode("overwrite") \
    .saveAsTable("catalog1.gold_layer.gold_fraud_table")